# Aula 01: Introdução aos Sistemas de Banco de Dados e Modelagem Conceitual
**Disciplina:** Banco de Dados  
**Curso:** Tecnologia em Telemática -- IFCE Campus Tauá  
**Professor:** Reginaldo Pereira Fernandes  

---
## 🎯 Objetivos da Aula
1. Compreender o papel de um Sistema Gerenciador de Banco de Dados (SGBD).
2. Identificar entidades, atributos e relacionamentos no mundo real.
3. Conectar e executar operações básicas em um banco relacional com Python (SQLite / PostgreSQL).

### 1. Criando e Conectando a um Banco Relacional em Python
Vamos criar tabelas para representar um cenário básico de telecomunicações (Dispositivos e Interfaces de Rede):

In [1]:
import sqlite3

# Conectando a um banco relacional em memória
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# DDL: Criação das tabelas com integridade referencial
cursor.execute("""
CREATE TABLE dispositivos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    hostname TEXT NOT NULL UNIQUE,
    tipo TEXT NOT NULL,
    localizacao TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE interfaces (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    dispositivo_id INTEGER NOT NULL,
    nome_interface TEXT NOT NULL,
    ip_address TEXT NOT NULL,
    mac_address TEXT NOT NULL UNIQUE,
    FOREIGN KEY (dispositivo_id) REFERENCES dispositivos(id)
);
""")

print("✅ Tabelas 'dispositivos' e 'interfaces' criadas com sucesso!")

### 2. Inserindo Registros e Consultando com JOIN
Agora inserimos dados e fazemos a junção relacional:

In [2]:
# Inserindo dispositivo de rede
cursor.execute("INSERT INTO dispositivos (hostname, tipo, localizacao) VALUES (?, ?, ?)", 
               ("SW-CORE-01", "Switch L3", "Rack 01 - Sala de Servidores"))
disp_id = cursor.lastrowid

# Inserindo interfaces vinculadas
interfaces = [
    (disp_id, "GigabitEthernet0/1", "192.168.1.1", "00:1A:2B:3C:4D:5E"),
    (disp_id, "GigabitEthernet0/2", "10.0.0.1", "00:1A:2B:3C:4D:5F")
]
cursor.executemany("INSERT INTO interfaces (dispositivo_id, nome_interface, ip_address, mac_address) VALUES (?, ?, ?, ?)", interfaces)
conn.commit()

# Consulta Relacional com JOIN
query = """
SELECT d.hostname, d.tipo, i.nome_interface, i.ip_address, i.mac_address
FROM dispositivos d
JOIN interfaces i ON i.dispositivo_id = d.id
"""

print(f"{'HOSTNAME':<14} {'TIPO':<12} {'INTERFACE':<20} {'IP':<15} {'MAC'}")
print("-" * 75)
for row in cursor.execute(query).fetchall():
    print(f"{row[0]:<14} {row[1]:<12} {row[2]:<20} {row[3]:<15} {row[4]}")